<a href="https://colab.research.google.com/github/spklapjs/Point-6/blob/main/ai_model/notebooks/02_model_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from google.colab import drive

# 1. 드라이브 마운트
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# 경로설정
BASE_DIR = '/content/drive/MyDrive/point-6/ai_model'
PROCESSED_DIR = os.path.join(BASE_DIR, 'data/processed')
CHECKPOINT_DIR = os.path.join(BASE_DIR, 'checkpoints')

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# 하이퍼파라미터 설정
BATCH_SIZE = 32
LEARNING_RATE = 0.001
EPOCHS = 50

In [3]:
# 2. 전처리된 텐서 데이터 로드 및 3분할 데이터로더 구성
features_tensor = torch.load(os.path.join(PROCESSED_DIR, 'features.pt'))
labels_tensor = torch.load(os.path.join(PROCESSED_DIR, 'labels.pt')).long()

dataset_size = len(features_tensor)
train_size = int(0.7 * dataset_size)
val_size = int(0.15 * dataset_size)
test_size = dataset_size - train_size - val_size

train_dataset, val_dataset, test_dataset = torch.utils.data.random_split(
    TensorDataset(features_tensor, labels_tensor),
    [train_size, val_size, test_size]
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"데이터셋 3분할 완료 - 학습: {train_size}개, 검증: {val_size}개, 테스트: {test_size}개")

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/point-6/ai_model/data/processed/features.pt'

In [ ]:
# 3. CNN-LSTM 하이브리드 모델 구조 정의
class CNNLSTMHybrid(nn.Module):
    def __init__(self, num_classes=6):
        super(CNNLSTMHybrid, self).__init__()

        # 공간적 특징 추출용 CNN
        self.cnn = nn.Sequential(
            nn.Conv1d(in_channels=8, out_channels=32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm1d(32),
            nn.Conv1d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm1d(64)
        )

        # 시간적 흐름 추출용 LSTM
        self.lstm = nn.LSTM(input_size=64, hidden_size=128, batch_first=True)

        # 최종 분류기
        self.fc = nn.Linear(128, num_classes)

    def forward(self, x):
        out = self.cnn(x)
        out = out.permute(0, 2, 1)
        out, _ = self.lstm(out)
        out = self.fc(out[:, -1, :])
        return out

model = CNNLSTMHybrid(num_classes=6)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

In [ ]:
# 4. 모델 학습, 최고 가중치 저장 및 최종 테스트 평가
best_accuracy = 0.0

print("모델 학습을 시작합니다...")
for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0

    for inputs, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in val_loader:
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {running_loss/len(train_loader):.4f}, Val Accuracy: {accuracy:.2f} 퍼센트")

    # 목표 정확도 관리를 위한 최고 성능 모델 pth 저장
    if accuracy > best_accuracy:
        best_accuracy = accuracy
        torch.save(model.state_dict(), os.path.join(CHECKPOINT_DIR, 'best_model.pth'))

print(f"학습 및 검증 완료. 최고 검증 정확도: {best_accuracy:.2f} 퍼센트")

In [ ]:
# 5. 분리된 테스트 데이터셋을 이용한 최종 모델 성능 평가
print("저장된 최고 성능 가중치로 최종 테스트 평가를 진행합니다...")
model.load_state_dict(torch.load(os.path.join(CHECKPOINT_DIR, 'best_model.pth')))
model.eval()

test_correct = 0
test_total = 0

with torch.no_grad():
    for inputs, labels in test_loader:
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        test_total += labels.size(0)
        test_correct += (predicted == labels).sum().item()

test_accuracy = 100 * test_correct / test_total
print(f"최종 테스트 정확도: {test_accuracy:.2f} 퍼센트")
print(f"최고 성능 가중치가 {CHECKPOINT_DIR}에 pth 형식으로 저장되었습니다.")